In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd
import numpy as np
from math import log, sqrt, erf
from datetime import datetime

import time

In [ ]:


def simulate_spread_strategy(
    starting_account=500,
    capital_at_risk_per_trade=100,
    risk_multiples=np.arange(1, 10.1, 0.25),
    win_rates=np.arange(0.50, 0.96, 0.02),
    num_trades=100,
    num_simulations=7000,
):
    results = []

    for risk_multiple in risk_multiples:
        # risk_multiple = max_loss / max_profit
        max_loss = capital_at_risk_per_trade
        max_profit = max_loss / risk_multiple

        for win_rate in win_rates:

            # create A 2D array of random numbers between 0 and 1
            random_matrix = np.random.random((num_simulations, num_trades))

            # If it’s < win_rate → WIN: assign +max_profit, Else → LOSS
            # NOTE: Numbers below win_rate represent wins”
            trade_pnl = np.where(
                random_matrix < win_rate,
                max_profit,
                -max_loss
            )

            final_pnl = trade_pnl.sum(axis=1)
            final_account = starting_account + final_pnl

            results.append({
                "Risk Multiple": round(risk_multiple, 2),
                "Win Rate": round(win_rate, 2),
                "Max Loss Per Trade": max_loss,
                "Max Profit Per Trade": round(max_profit, 2),
                "Mean Final Account": final_account.mean(),
                "Median Final Account": np.median(final_account),
                "Probability Profitable": (final_account > starting_account).mean(),
                "5th Percentile": np.percentile(final_account, 5),
                "95th Percentile": np.percentile(final_account, 95),
            })

    return pd.DataFrame(results)


# results = simulate_spread_strategy(
#     starting_account=500,
#     capital_at_risk_per_trade=100,
#     num_trades=100,
#     num_simulations=10
# )

# results.head()




### Data Ingestion

In [ ]:
# Parameters


TICKERS = [
    # your familiar names
    "HOOD", "PLTR", "ACHR", "MARA", "CLSK","PURR",
    # high-volume tech/growth you know
    # "NVDA", "AMD", "TSLA", "META", "GOOGL", "AMZN", "MSFT",
    "AAPL", "NFLX", "CRM", "SNOW", "COIN", "RBLX", "UBER",
    # robotics/AI theme for CSP discovery
    # "IONQ", "RGTI", "LUNR", "JOBY", "RIVN",
]

In [ ]:
# -----------------------------
# Helpers
# -----------------------------

def norm_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def estimate_put_delta(S, K, T, r, iv):
    """
    Black-Scholes put delta estimate.
    S = stock price
    K = strike
    T = time to expiration in years
    r = risk-free rate
    iv = implied volatility
    """
    if T <= 0 or iv <= 0:
        return None

    d1 = (log(S / K) + (r + 0.5 * iv**2) * T) / (iv * sqrt(T))
    put_delta = norm_cdf(d1) - 1
    return put_delta

def fetch_option_chain_with_backoff(ticker_obj, exp, max_retries=5, base_wait=1.23):
        retries = 0
        while retries < max_retries:
            try:
                chain = ticker_obj.option_chain(exp)
                puts = chain.puts.sort_values("strike", ascending=False).reset_index(drop=True)
                return puts
            except Exception as e:
                wait = base_wait * (2 ** retries)
                print(f"Error fetching option chain for {ticker_obj.ticker} {exp}, retry {retries+1}/{max_retries}. Reason: {e}. Waiting {wait:.1f}s before retry.")
                time.sleep(wait)
                retries += 1
        raise RuntimeError(f"Failed to fetch option chain for {ticker_obj.ticker} {exp} after {max_retries} retries.")


# -----------------------------
# Inputs
# -----------------------------

spread_width_target = 1
num_expirations = 5

max_loss_limit = 100
max_risk_multiple = 3.5
min_prob_profit = 0.70

risk_free_rate = 0.05  # rough estimate, can update later

# -----------------------------
# Pull data
# -----------------------------

df_list = []

for ticker in TICKERS:
    t = yf.Ticker(ticker)

    expirations = t.options[:num_expirations]
    current_price = t.history(period="1d")["Close"].iloc[-1]

    all_spreads = []

    # -----------------------------
    # Build bull put spreads
    # -----------------------------

    
    for exp in expirations:
        puts = fetch_option_chain_with_backoff(t, exp)
   
        exp_date = datetime.strptime(exp, "%Y-%m-%d")
        today = datetime.today()
        days_to_exp = (exp_date - today).days
        T = max(days_to_exp / 365, 1 / 365)

        for i in range(len(puts) - 1):
            sell = puts.iloc[i]       # higher strike put
            buy = puts.iloc[i + 1]    # lower strike put

            sell_strike = sell["strike"]
            buy_strike = buy["strike"]

            width = sell_strike - buy_strike

            # Only keep $1-wide spreads
            if abs(width - spread_width_target) > 0.01:
                continue

            # Use realistic execution:
            # sell at bid, buy at ask
            credit = sell["bid"] - buy["ask"]

            if credit <= 0:
                continue

            max_profit = credit * 100
            max_loss = (width - credit) * 100

            if max_profit <= 0:
                continue

            risk_multiple = max_loss / max_profit

            # Estimate probability using short put delta
            iv = sell.get("impliedVolatility", np.nan)

            if pd.isna(iv) or iv <= 0:
                short_put_delta = None
                prob_profit = None
            else:
                short_put_delta = estimate_put_delta(
                    S=current_price,
                    K=sell_strike,
                    T=T,
                    r=risk_free_rate,
                    iv=iv
                )

                # Approximation:
                # Sell put delta -0.30 ≈ 70% probability OTM
                prob_profit = 1 - abs(short_put_delta) if short_put_delta is not None else None

            all_spreads.append({
                "Ticker": ticker,
                "Current Price": round(current_price, 2),
                "Expiration": exp,
                "Days To Exp": days_to_exp,
                "Sell Strike": sell_strike,
                "Buy Strike": buy_strike,
                "Width": width,
                "Credit": round(credit, 2),
                "Max Profit": round(max_profit, 2),
                "Max Loss": round(max_loss, 2),
                "Risk Multiple": round(risk_multiple, 2),
                "Short Put IV": round(iv, 4) if not pd.isna(iv) else None,
                "Short Put Delta": round(short_put_delta, 3) if short_put_delta is not None else None,
                "Approx Prob Profit": round(prob_profit, 2) if prob_profit is not None else None,
                "Sell Bid": sell["bid"],
                "Buy Ask": buy["ask"],
                "Sell Volume": sell.get("volume", None),
                "Buy Volume": buy.get("volume", None),
                "Sell Open Interest": sell.get("openInterest", None),
                "Buy Open Interest": buy.get("openInterest", None),
            })

    df = pd.DataFrame(all_spreads)
    df_list.append(df)

all_options_data_for_ticker_df = pd.concat(df_list)
all_options_data_for_ticker_df


In [ ]:
# -----------------------------
# Filter results
# -----------------------------
max_risk_multiple = 5
filtered_df = all_options_data_for_ticker_df[
    (all_options_data_for_ticker_df["Max Loss"] <= max_loss_limit) &
    (all_options_data_for_ticker_df["Risk Multiple"] <= max_risk_multiple) &
    (all_options_data_for_ticker_df["Approx Prob Profit"] >= min_prob_profit) &
    (all_options_data_for_ticker_df['Days To Exp'].between(14, 40))
].copy()

filtered_df = filtered_df.sort_values(
    by=["Risk Multiple", "Approx Prob Profit"],
    ascending=[True, False]
)

# print("All spreads:")
# print(df.sort_values("Risk Multiple").head(2))

print("\nFiltered spreads:")
display(filtered_df.sort_values('Risk Multiple').head(20))